# Kien 0.86 Target-087 V2 Continuation

This notebook starts from the public Kien Tinker v14 adapter and makes a
small but more directed continuation update than the conservative build.

Safety choices:

- exact BF16 base model; no 4-bit quantization
- strict 100% warm-start weight coverage
- only `in_proj` LoRA tensors are trainable
- official-problem trajectories only
- 28 steps, peak LR `1.4e-6`, warmup, linear decay, gradient clipping
- retain 48% of the learned delta after training to limit regression
- keep the discovered corpus/order sequence instead of shuffling
- frozen Kien tensors and `lm_head` remain unchanged

The output is `submission.zip`. Keep `nemotron-087-kien-conservative.ipynb`
as the lower-risk fallback. A public score around 0.87 is a target, not a
guarantee; this configuration trades a little more movement for a better
chance of improving beyond the conservative run.


In [ ]:
# Conservative continuation config
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

MAX_SEQ_LEN = 8192
NUM_STEPS = 28
BATCH_SIZE = 32
MICRO_BATCH_SIZE = 4
LEARNING_RATE = 1.4e-6
WARMUP_STEPS = 3
GRAD_CLIP_NORM = 1.0
POST_TRAIN_DELTA_SCALE = 0.48

RESET_WEIGHTS = False
IN_PROJ_ONLY = True
MOE_TIE_WEIGHTS = False
ORIGINAL_PROBLEMS_ONLY = True
SHUFFLE_DATASET = False

KAGGLE_DATASET = "huikang/nemotron-data"
MINUTES = 60

# The full module set must match the warm-start adapter so every Kien tensor
# can be loaded. IN_PROJ_ONLY freezes all but the narrow continuation surface.
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "up_proj",
    "down_proj",
    "in_proj",
    "out_proj",
    "lm_head",
]


In [ ]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_MODAL_WORKER = "MODAL_TASK_ID" in os.environ
IS_MODAL_LAUNCHER = not IS_KAGGLE and not IS_MODAL_WORKER

In [ ]:
# -- Env-specific install (Kaggle only; Modal image has packages pre-installed) --
if IS_KAGGLE:
    import subprocess

    from pathlib import Path

    INPUT_ROOT = Path("/kaggle/input")

    def list_input_tree(root, max_depth=4, limit=250):
        root = Path(root)
        rows = []
        stack = [(root, 0)]
        while stack and len(rows) < limit:
            base, depth = stack.pop()
            if depth >= max_depth:
                continue
            try:
                children = sorted(base.iterdir(), key=lambda path: path.name)
            except OSError:
                continue
            for child in children:
                rows.append(str(child))
                if child.is_dir():
                    stack.append((child, depth + 1))
                if len(rows) >= limit:
                    break
        return rows

    def find_path(label, candidates, predicate=lambda path: True, required=True):
        import glob

        matches = []
        for candidate in candidates:
            candidate = Path(candidate)
            if any(char in str(candidate) for char in "*?["):
                matches.extend(
                    path
                    for path in map(
                        Path,
                        glob.glob(
                            str(candidate),
                            recursive="**" in str(candidate),
                        ),
                    )
                    if predicate(path)
                )
            elif predicate(candidate):
                matches.append(candidate)
        matches = sorted(set(path.resolve() for path in matches))
        if not matches:
            message = (
                f"{label} not found using candidates {[str(x) for x in candidates]}. "
                f"/kaggle/input tree sample: {list_input_tree(INPUT_ROOT)}"
            )
            if required:
                raise AssertionError(message)
            print(f"Optional input skipped: {message}")
            return None
        if len(matches) > 1:
            preferred = [
                path
                for path in matches
                if any(
                    token in str(path).lower()
                    for token in (
                        "tinker-adapter",
                        "nemotron-3-nano-30b-a3b-bf16",
                        "huikang-nemotron-repository-snapshot",
                        "nvidia-nemotron-model-reasoning-challenge",
                    )
                )
            ]
            if len(preferred) >= 1:
                matches = preferred
            if len(matches) > 1 and not required:
                print(
                    f"Optional {label} matched multiple paths; using {matches[0]}. "
                    f"All matches: {[str(path) for path in matches]}"
                )
                matches = [matches[0]]
        assert len(matches) == 1, (
            f"{label} is ambiguous; matches: {[str(path) for path in matches]}"
        )
        return matches[0]

    def find_unique(label, candidates, predicate=lambda path: True):
        return find_path(label, candidates, predicate, required=True)

    def find_optional(label, candidates, predicate=lambda path: True):
        return find_path(label, candidates, predicate, required=False)

    PACKAGES_DIR = find_unique(
        "nemotron offline packages",
        [
            "/kaggle/input/datasets/mayukh18/nemotron-packages/packages",
            "/kaggle/input/datasets/mayukh18/nemotron-packages/*/packages",
        ],
        lambda path: path.is_dir(),
    )
    CAUSAL_WHEEL = find_unique(
        "causal_conv1d wheel",
        [
            "/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-*.whl",
            "/kaggle/input/datasets/mayukh18/nemotron-packages/*/causal_conv1d-*.whl",
        ],
        lambda path: path.is_file(),
    )
    MAMBA_WHEEL = find_unique(
        "mamba_ssm wheel",
        [
            "/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-*.whl",
            "/kaggle/input/datasets/mayukh18/nemotron-packages/*/mamba_ssm-*.whl",
        ],
        lambda path: path.is_file(),
    )
    RTX_WHEELS_DIR = find_unique(
        "RTX wheels directory",
        [
            "/kaggle/input/datasets/llkh0a/rtx-wheels/wheels",
            "/kaggle/input/datasets/llkh0a/rtx-wheels/*/wheels",
        ],
        lambda path: path.is_dir(),
    )
    MODEL_CONFIG = find_unique(
        "Nemotron BF16 base model",
        [
            "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1/config.json",
            "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/*/config.json",
        ],
        lambda path: path.is_file()
        and (
            (path.parent / "model.safetensors.index.json").is_file()
            or any(path.parent.glob("*.safetensors"))
        ),
    )
    MODEL_PATH = str(MODEL_CONFIG.parent)
    ADAPTER_CONFIG = find_unique(
        "Kien tinker adapter",
        [
            "/kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1/adapter_config.json",
            "/kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/*/adapter_config.json",
        ],
        lambda path: path.is_file()
        and (path.parent / "adapter_model.safetensors").is_file(),
    )
    ADAPTER_SRC = str(ADAPTER_CONFIG.parent)
    with ADAPTER_CONFIG.open() as config_file:
        discovered_adapter_config = __import__("json").load(config_file)
    assert discovered_adapter_config.get("r") == LORA_RANK
    assert discovered_adapter_config.get("lora_alpha") == LORA_ALPHA
    adapter_bytes = (ADAPTER_CONFIG.parent / "adapter_model.safetensors").stat().st_size
    assert adapter_bytes > 3_000_000_000, (
        f"Unexpectedly small Kien adapter: {adapter_bytes:,} bytes"
    )
    TRAIN_CSV = find_unique(
        "competition train.csv",
        [
            "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv",
        ],
        lambda path: path.is_file(),
    )
    TRAIN_CSV_PATH = str(TRAIN_CSV)
    with TRAIN_CSV.open(encoding="utf-8") as train_file:
        train_header = train_file.readline().strip().split(",")
    assert train_header == ["id", "prompt", "answer"], train_header
    HUIKANG_SFT_DIR = find_optional(
        "Huikang 04-08-16-14 corpus",
        [
            "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/nemotron-master/training/sft/04-08-16-14",
            "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/*/nemotron-master/training/sft/04-08-16-14",
            "/kaggle/input/huikang-nemotron-repository-snapshot/nemotron-master/training/sft/04-08-16-14",
            "/kaggle/input/datasets/huikang/*/nemotron-master/training/sft/04-08-16-14",
            "/kaggle/input/datasets/huikang/*/*/nemotron-master/training/sft/04-08-16-14",
            "/kaggle/input/datasets/huikang/**/nemotron-master/training/sft/04-08-16-14",
        ],
        lambda path: path.is_dir()
        and (path / "tokens").is_dir()
        and (path / "logprobs/index.jsonl").is_file(),
    )
    if HUIKANG_SFT_DIR is not None:
        CORPUS_PATH = str(HUIKANG_SFT_DIR / "tokens")
        TRAIN_ORDER_PATH = str(HUIKANG_SFT_DIR / "logprobs/index.jsonl")
    else:
        CORPUS_PATH = None
        TRAIN_ORDER_PATH = None

    discovered_inputs = {
        "packages": PACKAGES_DIR,
        "causal wheel": CAUSAL_WHEEL,
        "mamba wheel": MAMBA_WHEEL,
        "RTX wheels": RTX_WHEELS_DIR,
        "base model": MODEL_PATH,
        "Kien adapter": ADAPTER_SRC,
        "train.csv": TRAIN_CSV_PATH,
    }
    if CORPUS_PATH is not None:
        discovered_inputs["Huikang tokens"] = CORPUS_PATH
        discovered_inputs["Huikang order"] = TRAIN_ORDER_PATH
    else:
        discovered_inputs["Huikang corpus"] = "not mounted; using train.csv fallback"

    print("Mounted Kaggle Inputs were discovered and validated.")
    for label, path in discovered_inputs.items():
        print(f"  {label}: {path}")

    subprocess.run(
        f"pip install -q --no-index --find-links {PACKAGES_DIR} "
        "unsloth trl peft transformers datasets accelerate bitsandbytes",
        shell=True,
        check=True,
    )
    subprocess.run(
        f"pip install -q {CAUSAL_WHEEL}",
        shell=True,
        check=True,
    )
    subprocess.run(
        f"pip install -q {MAMBA_WHEEL}",
        shell=True,
        check=True,
    )
    for _wd in [str(RTX_WHEELS_DIR)]:
        if os.path.isdir(_wd):
            subprocess.run(
                [
                    "pip",
                    "install",
                    "-q",
                    "--no-index",
                    "--find-links",
                    _wd,
                    "protobuf==6.33.5",
                    "sentencepiece",
                    "safetensors",
                    "huggingface_hub",
                ],
                check=False,
            )
    subprocess.run("rm -rf /kaggle/tmp/*", shell=True, check=True)

In [ ]:
def run_training() -> None:
    """Full training flow. Runs on Kaggle at module level or inside Modal container via train_remote()."""
    import gc
    import json
    import math
    import random
    import subprocess
    import sys
    import time

    from unsloth import FastLanguageModel

    import torch
    from cut_cross_entropy import linear_cross_entropy
    from peft import LoraConfig
    from peft.tuners.lora import Linear as LoraLinear

    # -- Env-specific paths + adapter source --------------------------
    if IS_KAGGLE:
        CORPUS_PATH = globals().get("CORPUS_PATH")
        TRAIN_ORDER_PATH = globals().get("TRAIN_ORDER_PATH")
        TRAIN_CSV_PATH = globals()["TRAIN_CSV_PATH"]
        ADAPTER_SRC = globals()["ADAPTER_SRC"]
        MODEL_PATH = globals()["MODEL_PATH"]
        adapter_config_path = os.path.join(ADAPTER_SRC, "adapter_config.json")
        adapter_weights_path = os.path.join(
            ADAPTER_SRC, "adapter_model.safetensors"
        )
        assert os.path.isfile(adapter_config_path), adapter_config_path
        assert os.path.isfile(adapter_weights_path), adapter_weights_path
        with open(adapter_config_path) as _f:
            warmstart_config = json.load(_f)
        assert warmstart_config.get("r") == LORA_RANK, warmstart_config
        assert warmstart_config.get("lora_alpha") == LORA_ALPHA, warmstart_config
        print(
            "Warm start: Kien Tinker v14, "
            f"rank={warmstart_config.get('r')}, "
            f"alpha={warmstart_config.get('lora_alpha')}"
        )
    else:  # IS_MODAL_WORKER
        MODEL_PATH = "unsloth/Nemotron-3-Nano-30B-A3B"
        CORPUS_PATH = "/data/corpus_preprocessed.jsonl"
        TRAIN_CSV_PATH = "/data/train.csv"
        ADAPTER_SRC = "/merged/weights"
        OUTPUT_DIR = "/output/weights"

    # -- GPU + kernel sanity check (runs on both Kaggle and Modal worker) --
    import causal_conv1d
    import mamba_ssm

    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}, sm_{cc[0] * 10 + cc[1]}")
    print(f"torch={torch.__version__}, cuda={torch.version.cuda}")
    print(
        f"mamba_ssm={mamba_ssm.__version__}, causal_conv1d={causal_conv1d.__version__}"
    )
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if IS_MODAL_WORKER:
        assert cc == (12, 0), (
            f"Expected sm_120 (RTX PRO 6000), got sm_{cc[0] * 10 + cc[1]}"
        )
    from causal_conv1d import causal_conv1d_fn

    _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
    _w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
    causal_conv1d_fn(_x, _w, None, activation="silu")
    print("causal_conv1d CUDA kernel: OK")

    # Clear stale HF modules cache (Modal-only; bug: persists across runs)
    if IS_MODAL_WORKER:
        import shutil as _shutil

        hf_modules = os.path.join(
            os.environ.get("HF_HOME", "/root/.cache/huggingface"), "modules"
        )
        if os.path.exists(hf_modules):
            _shutil.rmtree(hf_modules)

    # -- Load corpus into `examples` list -----------------------------
    examples: list[dict] = []

    if IS_KAGGLE and CORPUS_PATH and TRAIN_ORDER_PATH:
        # Load problem_ids in training order from logprobs/index.jsonl (epoch 0).
        # Each entry has {epoch, step, problem_id, ...}; epoch-0 entries are the
        # original training order, which we replay here.
        ordered_ids: list[str] = []
        seen: set[str] = set()
        with open(TRAIN_ORDER_PATH) as f:
            for line in f:
                rec = json.loads(line)
                if rec.get("epoch", 0) != 0:
                    continue
                pid = rec["problem_id"]
                if pid in seen:
                    continue
                seen.add(pid)
                ordered_ids.append(pid)
        print(
            f"Loaded {len(ordered_ids)} problem_ids in training order from "
            f"{TRAIN_ORDER_PATH}"
        )

        for sid in ordered_ids:
            seg_path = os.path.join(CORPUS_PATH, sid, "synthetic.json")
            assert os.path.isfile(seg_path), (
                f"problem_id {sid} from training order missing in corpus: {seg_path}"
            )
            with open(seg_path) as f:
                rec = json.load(f)
            tokens = rec["tokens"]
            mask = rec["mask"]
            if not tokens:
                continue
            if len(tokens) > MAX_SEQ_LEN:
                tokens = tokens[:MAX_SEQ_LEN]
                mask = mask[:MAX_SEQ_LEN]
            if not any(mask):
                continue
            examples.append(
                {
                    "problem_id": sid,
                    "tokens": tokens[:-1],
                    "targets": tokens[1:],
                    "weights": [float(m) for m in mask[1:]],
                }
            )
    elif IS_KAGGLE:
        print(
            "Huikang token corpus is not mounted; "
            "will tokenize competition train.csv after tokenizer load."
        )
    else:  # IS_MODAL_WORKER
        with open(CORPUS_PATH) as f:
            for line in f:
                rec = json.loads(line.strip())
                tokens = rec["tokens"]
                mask = rec["mask"]
                if len(tokens) > MAX_SEQ_LEN:
                    tokens = tokens[:MAX_SEQ_LEN]
                    mask = mask[:MAX_SEQ_LEN]
                if not any(mask):
                    continue
                examples.append(
                    {
                        "problem_id": rec["problem_id"],
                        "tokens": tokens[:-1],
                        "targets": tokens[1:],
                        "weights": [float(m) for m in mask[1:]],
                    }
                )

    if ORIGINAL_PROBLEMS_ONLY:
        import csv

        with open(TRAIN_CSV_PATH) as f:
            original_ids = {row["id"] for row in csv.DictReader(f)}
        before = len(examples)
        examples = [e for e in examples if e["problem_id"] in original_ids]
        print(
            f"ORIGINAL_PROBLEMS_ONLY=True: filtered {before} -> {len(examples)} examples "
            f"using {len(original_ids)} ids from {TRAIN_CSV_PATH}"
        )

    total_unmasked = sum(sum(e["weights"]) for e in examples)
    total_tokens = sum(len(e["tokens"]) for e in examples)
    print(
        f"Loaded {len(examples)} examples, {total_tokens:,} tokens "
        f"(unmasked={total_unmasked:,.0f})"
    )

    # -- Load base model ----------------------------------------------
    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=True,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if IS_MODAL_WORKER:
        hf_cache_vol.commit()  # noqa: F821 - defined at module level on non-Kaggle

    if IS_KAGGLE and not examples:
        import csv

        def _tokenize_train_row(row: dict) -> dict | None:
            prompt = str(row.get("prompt", ""))
            answer = "\n" + str(row.get("answer", ""))
            prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
            if tokenizer.eos_token_id is not None:
                answer_ids = answer_ids + [tokenizer.eos_token_id]
            tokens = prompt_ids + answer_ids
            mask = [0] * len(prompt_ids) + [1] * len(answer_ids)
            if len(tokens) > MAX_SEQ_LEN:
                overflow = len(tokens) - MAX_SEQ_LEN
                if overflow < len(prompt_ids):
                    prompt_ids = prompt_ids[overflow:]
                    tokens = prompt_ids + answer_ids
                    mask = [0] * len(prompt_ids) + [1] * len(answer_ids)
                else:
                    tokens = tokens[-MAX_SEQ_LEN:]
                    mask = mask[-MAX_SEQ_LEN:]
            if len(tokens) < 2 or not any(mask[1:]):
                return None
            return {
                "problem_id": str(row.get("id", "")),
                "tokens": tokens[:-1],
                "targets": tokens[1:],
                "weights": [float(m) for m in mask[1:]],
            }

        with open(TRAIN_CSV_PATH, encoding="utf-8", newline="") as f:
            for row in csv.DictReader(f):
                example = _tokenize_train_row(row)
                if example is not None:
                    examples.append(example)
        total_unmasked = sum(sum(e["weights"]) for e in examples)
        total_tokens = sum(len(e["tokens"]) for e in examples)
        print(
            f"train.csv fallback loaded {len(examples)} examples, "
            f"{total_tokens:,} tokens (unmasked={total_unmasked:,.0f})"
        )

    # -- Wrap in LoRA -------------------------------------------------
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    FastLanguageModel.for_training(model)

    # -- Patch Mamba CUDA fast path -----------------------------------
    nemotron_mod = None
    for _name, _m in sys.modules.items():
        if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
            nemotron_mod = _m
            break
    assert nemotron_mod is not None, "Could not find modeling_nemotron_h module"
    print(f"is_fast_path_available was: {nemotron_mod.is_fast_path_available}")
    nemotron_mod.is_fast_path_available = True  # type: ignore[attr-defined]
    print("Patched is_fast_path_available = True")

    # -- Manually add lm_head LoRA (Unsloth drops it for MoE) ---------
    _causal_lm = model
    while hasattr(_causal_lm, "model"):
        _causal_lm = _causal_lm.model
    _lm_head = _causal_lm.lm_head
    if not isinstance(_lm_head, LoraLinear):
        _cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(
            _cfg,
            "default",
            target=_lm_head,
            target_name="lm_head",
            parent=_causal_lm,
        )
        print("Manually added LoRA to lm_head")
    else:
        print("lm_head already has LoRA")

    # -- Cast LoRA params to fp32 (base model stays bf16 except MoE router) --
    for name, param in model.named_parameters():
        if ".lora_" in name:
            param.data = param.data.to(torch.float32)

    for name, param in model.named_parameters():
        if ".lora_" in name:
            assert param.dtype == torch.float32, (
                f"LoRA param {name} expected fp32, got {param.dtype}"
            )
            continue

        is_router = (
            ".mixer.gate." in name
        )  # NemotronHTopkRouter.weight + e_score_correction_bias
        # Nemotron-H loads the MoE router (`mixer.gate`) in fp32 on purpose.
        # Ref: transformers/src/transformers/models/nemotron_h/modeling_nemotron_h.py
        #
        #   class NemotronHPreTrainedModel(PreTrainedModel):
        #       _keep_in_fp32_modules_strict = ["e_score_correction_bias"]
        #
        #   class NemotronHTopkRouter(nn.Module):
        #       def __init__(self, config):
        #           self.weight = nn.Parameter(torch.empty((self.n_routed_experts, config.hidden_size)))
        #           self.register_buffer("e_score_correction_bias", torch.zeros(self.n_routed_experts))
        #       def forward(self, hidden_states):
        #           router_logits = F.linear(
        #               hidden_states.type(torch.float32),
        #               self.weight.type(torch.float32),
        #           )
        #           return router_logits
        #
        # The per-forward fp32 cast on `self.weight` plus the strict list entry
        # mean the gate weight is promoted to fp32 at load time.
        if is_router:
            assert param.dtype == torch.float32, (
                f"param {name} expected fp32, got {param.dtype}"
            )
            continue

        assert param.dtype in [torch.bfloat16, torch.float16, torch.float32], (
            f"param {name} expected bf16/fp16/fp32, got {param.dtype}"
        )
        continue

    print("Verified: LoRA params fp32, base params bf16 (MoE router fp32)")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Model: {trainable:,} trainable / {total:,} total parameters")

    # -- Patch forward with Cut Cross-Entropy -------------------------
    _base = model
    while hasattr(_base, "model"):
        _base = _base.model

    def _patched_causal_forward(
        input_ids=None, attention_mask=None, labels=None, **kwargs
    ):
        backbone_out = _base.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **{
                k: v
                for k, v in kwargs.items()
                if k in ("position_ids", "past_key_values", "use_cache")
            },
        )
        hidden_states = backbone_out[0]
        lm_head = _base.lm_head
        base_w = lm_head.base_layer.weight
        lora_A = lm_head.lora_A["default"].weight
        lora_B = lm_head.lora_B["default"].weight
        scaling = lm_head.scaling["default"]
        lm_weight = base_w + scaling * lora_B @ lora_A
        if labels is not None:
            per_token_ce = linear_cross_entropy(
                hidden_states, lm_weight, labels, reduction="none"
            )
            loss = per_token_ce.mean()
        else:
            per_token_ce = None
            loss = None
        model._cached_per_token_ce = per_token_ce  # type: ignore[attr-defined]
        return loss

    _base.forward = _patched_causal_forward
    print("Patched CausalLM.forward with CCE (no logits materialization)")

    # -- Load adapter weights (unless RESET_WEIGHTS) ------------------
    if RESET_WEIGHTS:
        print(
            "RESET_WEIGHTS=True - skipping pretrained adapter load; using fresh LoRA init"
        )
        loaded = 0
        adapter_weights: dict = {}
        loaded_model_keys: set[str] = set()
    else:
        print(f"Loading adapter from {ADAPTER_SRC}...")
        from peft import load_peft_weights

        adapter_weights = load_peft_weights(ADAPTER_SRC)
        warmstart_in_proj = [
            key for key in adapter_weights if ".in_proj." in key
        ]
        assert warmstart_in_proj, (
            "Kien adapter has no in_proj tensors; refusing a fresh random update"
        )

        import re
        from collections import defaultdict

        model_sd = model.state_dict()
        new_sd: dict = {}
        consumed_source_keys: set[str] = set()

        def lora_side(key: str) -> str:
            if ".lora_A." in key:
                return "A"
            if ".lora_B." in key:
                return "B"
            raise ValueError(key)

        def source_tail(key: str) -> str:
            key = key.replace(".lora_A.weight", ".lora_A.default.weight")
            key = key.replace(".lora_B.weight", ".lora_B.default.weight")
            layer_match = re.search(r"(layers\.\d+\..*)", key)
            if layer_match:
                return layer_match.group(1)
            lm_match = re.search(r"(lm_head\..*)", key)
            if lm_match:
                return lm_match.group(1)
            return key

        def unique_shape_match(
            source_key: str,
            tensor_variants: list[tuple[str, torch.Tensor]],
            candidates: list[str],
        ) -> tuple[str, torch.Tensor, str]:
            shape_matches = [
                (key, tensor, variant_name)
                for variant_name, tensor in tensor_variants
                for key in candidates
                if tuple(model_sd[key].shape) == tuple(tensor.shape)
            ]
            assert len(shape_matches) == 1, (
                f"Expected one model layout for {source_key}; "
                f"variants={[(name, tuple(t.shape)) for name, t in tensor_variants]}, "
                f"candidates={[(key, tuple(model_sd[key].shape)) for key in candidates[:20]]}, "
                f"matches={[(key, name) for key, _, name in shape_matches[:20]]}"
            )
            return shape_matches[0]

        expert_pattern = re.compile(
            r"^(.*?layers\.(\d+)\..*?\.experts\.)(\d+)\."
            r"(up_proj|down_proj)(\.lora_([AB])\.weight)$"
        )
        expert_groups: dict[tuple, list[tuple[int, str, torch.Tensor]]] = (
            defaultdict(list)
        )

        # Non-expert tensors retain their projection names. Match by the
        # layer-relative suffix so backbone/model wrapper prefixes may differ.
        for source_key, tensor in adapter_weights.items():
            match = expert_pattern.match(source_key)
            if match:
                group_key = (
                    match.group(1),
                    int(match.group(2)),
                    match.group(4),
                    match.group(6),
                )
                expert_groups[group_key].append(
                    (int(match.group(3)), source_key, tensor)
                )
                continue

            tail = source_tail(source_key)
            candidates = [
                key
                for key in model_sd
                if key.endswith(tail)
                and lora_side(key) == lora_side(source_key)
            ]
            if not candidates:
                layer_match = re.search(r"layers\.(\d+)\.", source_key)
                projection_match = re.search(
                    r"\.(in_proj|out_proj|q_proj|k_proj|v_proj|o_proj|"
                    r"up_proj|down_proj)\.lora_[AB]\.weight$",
                    source_key,
                )
                candidates = [
                    key
                    for key in model_sd
                    if ".lora_" in key
                    and (
                        layer_match is None
                        or f"layers.{layer_match.group(1)}." in key
                    )
                    and (
                        projection_match is None
                        or f".{projection_match.group(1)}." in key
                    )
                    and (
                        "shared_experts" not in source_key
                        or ".shared_experts." in key
                    )
                    and lora_side(key) == lora_side(source_key)
                ]
            target_key, converted_tensor, _ = unique_shape_match(
                source_key,
                [("identity", tensor)],
                candidates,
            )
            new_sd[target_key] = converted_tensor
            consumed_source_keys.add(source_key)

        # In this Unsloth build, expert tensors stay one-per-expert. Convert
        # every source key individually; keep folded/batched fallback only for
        # other builds.
        for (_, layer, projection, side), entries in expert_groups.items():
            entries.sort(key=lambda row: row[0])
            expert_ids = [row[0] for row in entries]
            assert expert_ids == list(range(128)), (
                f"Layer {layer} {projection} LoRA-{side}: "
                f"expected experts 0..127, got {expert_ids[:10]}..."
            )
            loaded_direct_experts = 0
            projection_names = (
                ("up_proj", "gate_up_proj")
                if projection == "up_proj"
                else ("down_proj",)
            )
            for expert_id, source_key, tensor in entries:
                candidates = [
                    key
                    for key in model_sd
                    if f"layers.{layer}." in key
                    and f".experts.{expert_id}." in key
                    and ".lora_" in key
                    and any(f".{name}." in key for name in projection_names)
                    and lora_side(key) == side
                ]
                target_key, converted_tensor, layout = unique_shape_match(
                    source_key,
                    [("identity", tensor)],
                    candidates,
                )
                new_sd[target_key] = converted_tensor
                consumed_source_keys.add(source_key)
                loaded_direct_experts += 1
            print(
                f"    layer={layer} {projection} LoRA-{side}: "
                f"loaded {loaded_direct_experts} per-expert tensors"
            )

        missing_source_keys = sorted(
            set(adapter_weights) - consumed_source_keys
        )
        assert not missing_source_keys, (
            f"Unconverted Kien tensors: {missing_source_keys[:20]}"
        )
        assert len(consumed_source_keys) == len(adapter_weights), (
            f"Not all Kien weights converted: "
            f"{len(consumed_source_keys)}/{len(adapter_weights)}"
        )
        incompatible = model.load_state_dict(new_sd, strict=False)
        loaded_model_keys = set(new_sd)
        print(
            f"  Converted and loaded {len(consumed_source_keys)}/"
            f"{len(adapter_weights)} Kien tensors into "
            f"{len(new_sd)} Unsloth tensors"
        )
        print(
            f"  load_state_dict: missing={len(incompatible.missing_keys)}, "
            f"unexpected={len(incompatible.unexpected_keys)}"
        )

    # -- Freeze all LoRA params except in_proj (if IN_PROJ_ONLY) --
    print(f"{IN_PROJ_ONLY=}")
    if IN_PROJ_ONLY:
        for name, param in model.named_parameters():
            if param.requires_grad and ".in_proj." not in name:
                param.requires_grad = False
    trainable_names = [
        name for name, param in model.named_parameters() if param.requires_grad
    ]
    assert trainable_names, "No trainable LoRA parameters found"
    if IN_PROJ_ONLY:
        unexpected_trainable = [
            name for name in trainable_names if ".in_proj." not in name
        ]
        assert not unexpected_trainable, unexpected_trainable[:20]
        missing_warmstart = [
            name
            for name in trainable_names
            if name not in loaded_model_keys
            and f"{name}.weight" not in loaded_model_keys
        ]
        assert not missing_warmstart, (
            "Trainable tensors missing from Kien warm start: "
            f"{missing_warmstart[:20]}"
        )
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  {trainable_params:,} trainable / {frozen_params:,} frozen")
    print(f"  trainable tensors: {len(trainable_names)}")
    initial_trainable = {
        name: param.detach().cpu().clone()
        for name, param in model.named_parameters()
        if param.requires_grad
    }
    print(
        f"  snapshotted {len(initial_trainable)} trainable tensors "
        "for post-training delta scaling"
    )

    # -- MoE tied-weight params (Tinker convention) -------------------
    # Tinker ties whichever LoRA side touches the hidden dim:
    #   gate_up_proj / up_proj / w1 / gate_proj  -> tie A (input/hidden side)
    #   down_proj / w2                           -> tie B (output/hidden side)
    # We keep Unsloth's batched [num_experts, ...] tensor layout; "tying" means
    # all 128 expert slices are kept identical. Saving the adapter naturally
    # emits 128 per-expert copies, so submission.zip is untied downstream.
    moe_tied_params: list[torch.Tensor] = []
    if MOE_TIE_WEIGHTS:
        w1_proj_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
        w2_proj_names = ("down_proj", ".w2.")
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if ".experts." not in name or ".lora_" not in name:
                continue
            is_w1 = any(p in name for p in w1_proj_names)
            is_w2 = any(p in name for p in w2_proj_names)
            is_A = ".lora_A." in name
            is_B = ".lora_B." in name
            should_tie = (is_w1 and is_A) or (is_w2 and is_B)
            if not should_tie:
                continue
            if param.dim() < 2 or param.shape[0] <= 1:
                continue
            moe_tied_params.append(param)

        def _tie_param_init() -> None:
            """Make all 128 expert slices identical (mean-and-broadcast)."""
            with torch.no_grad():
                for p in moe_tied_params:
                    mean = p.data.mean(dim=0, keepdim=True)
                    p.data.copy_(mean.expand_as(p.data))

        def _tie_grads() -> None:
            # Sum (not mean) across the expert dim: if W is the shared LoRA factor
            # and each expert uses a copy W_i = W, chain rule gives
            # dL/dW = sum_i dL/dW_i. Inactive experts contribute 0 and router
            # weights are already baked into active g_i, so there's no
            # double-counting. Summing keeps all 128 slices identical after each
            # AdamW step and reproduces the true shared-weight update; mean would
            # be off by a 1/128 lr rescale (and not exactly equivalent under
            # AdamW's eps/weight-decay).
            with torch.no_grad():
                for p in moe_tied_params:
                    if p.grad is None:
                        continue
                    grad_sum = p.grad.sum(dim=0, keepdim=True)
                    p.grad.copy_(grad_sum.expand_as(p.grad))

        print(f"MoE weight tying: {len(moe_tied_params)} params identified for tying")
        if moe_tied_params:
            print(f"  example shapes: {[tuple(p.shape) for p in moe_tied_params[:3]]}")
        _tie_param_init()  # start from a tied state
    else:

        def _tie_grads() -> None:
            pass

    # -- Training loop ------------------------------------------------
    gc.collect()
    torch.cuda.empty_cache()

    device = next(model.parameters()).device
    optimizer: torch.optim.AdamW | None = None

    assert examples, (
        "No training examples loaded. Mount the Huikang token corpus or ensure "
        "competition train.csv has prompt/answer rows."
    )
    indices = list(range(len(examples)))
    if SHUFFLE_DATASET:
        rng = random.Random(0)
        rng.shuffle(indices)
        print(f"SHUFFLE_DATASET=True: shuffled {len(indices)} examples (seed=0)")
    else:
        print(f"SHUFFLE_DATASET=False: keeping corpus order ({len(indices)} examples)")

    training_log: list[str] = []

    def _log(msg: str) -> None:
        print(msg, flush=True)
        training_log.append(msg)

    max_steps = math.ceil(len(examples) / BATCH_SIZE)
    num_steps = NUM_STEPS
    if num_steps > max_steps:
        _log(
            f"WARNING: NUM_STEPS={NUM_STEPS} exceeds max_steps={max_steps} "
            f"({len(examples)} examples // {BATCH_SIZE} batch). Clamping to {max_steps}."
        )
        num_steps = max_steps

    _log(
        f"Training: {num_steps} steps, batch_size={BATCH_SIZE}, "
        f"micro_batch_size={MICRO_BATCH_SIZE}, lr={LEARNING_RATE}"
    )

    step = 0
    for batch_start in range(0, len(indices), BATCH_SIZE):
        if step >= num_steps:
            break
        batch_indices = indices[batch_start : batch_start + BATCH_SIZE]
        batch = [examples[i] for i in batch_indices]
        batch_tokens = [e["tokens"] for e in batch]
        batch_targets = [e["targets"] for e in batch]
        batch_weights = [e["weights"] for e in batch]

        n = len(batch)
        n_accum = math.ceil(n / MICRO_BATCH_SIZE)
        total_loss_sum = 0.0
        total_weight_sum = 0.0

        for mb_start in range(0, n, MICRO_BATCH_SIZE):
            mb_end = min(mb_start + MICRO_BATCH_SIZE, n)
            mb_toks = batch_tokens[mb_start:mb_end]
            mb_tgts = batch_targets[mb_start:mb_end]
            mb_wts = batch_weights[mb_start:mb_end]

            n_micro = len(mb_toks)
            max_len = max(len(t) for t in mb_toks)
            total_len = sum(len(t) for t in mb_toks)

            padded_input = torch.zeros(
                n_micro, max_len, dtype=torch.long, device=device
            )
            padded_targets = torch.zeros(
                n_micro, max_len, dtype=torch.long, device=device
            )
            padded_weights = torch.zeros(
                n_micro, max_len, dtype=torch.float32, device=device
            )
            attention_mask = torch.zeros(
                n_micro, max_len, dtype=torch.long, device=device
            )
            for i in range(n_micro):
                seq_len = len(mb_toks[i])
                padded_input[i, :seq_len] = torch.tensor(mb_toks[i], dtype=torch.long)
                padded_targets[i, :seq_len] = torch.tensor(mb_tgts[i], dtype=torch.long)
                padded_weights[i, :seq_len] = torch.tensor(
                    mb_wts[i], dtype=torch.float32
                )
                attention_mask[i, :seq_len] = 1

            t0 = time.time()
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                model(
                    input_ids=padded_input,
                    attention_mask=attention_mask,
                    labels=padded_targets,
                    use_cache=False,
                )
                per_token_ce = model._cached_per_token_ce  # type: ignore[attr-defined]
                weighted_loss = per_token_ce * padded_weights
                weight_sum_t = padded_weights.sum()
                loss_sum_t = weighted_loss.sum()
                loss = (
                    loss_sum_t / weight_sum_t if weight_sum_t > 0 else loss_sum_t * 0.0
                )
                assert torch.isfinite(loss), f"Non-finite loss at step {step}"

            (loss / n_accum).backward()
            total_loss_sum += loss_sum_t.item()
            total_weight_sum += weight_sum_t.item()
            del loss, per_token_ce, weighted_loss

            t_end = time.time()
            peak_gb = torch.cuda.max_memory_allocated() / 1e9
            mem_gb = torch.cuda.memory_allocated() / 1e9
            mb_idx = mb_start // MICRO_BATCH_SIZE
            print(
                f"    micro-batch {mb_idx}: {n_micro} seqs, max_len={max_len}, "
                f"total_len={total_len}, wall={t_end - t0:.1f}s, "
                f"peak={peak_gb:.1f}GB, mem={mem_gb:.1f}GB"
            )

        if optimizer is None:
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=LEARNING_RATE,
                betas=(0.9, 0.95),
                eps=1e-8,
                weight_decay=0.0,
            )
        if step < WARMUP_STEPS:
            lr = LEARNING_RATE * (step + 1) / WARMUP_STEPS
        else:
            decay_steps = max(1, num_steps - WARMUP_STEPS)
            decay_progress = (step - WARMUP_STEPS) / decay_steps
            lr = LEARNING_RATE * max(0.0, 1.0 - decay_progress)
        for pg in optimizer.param_groups:
            pg["lr"] = lr
        _tie_grads()  # average MoE expert grads before clip+step so Adam stays in sync
        grad_norm = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=GRAD_CLIP_NORM,
        )
        optimizer.step()
        optimizer.zero_grad()
        loss_mean = total_loss_sum / total_weight_sum if total_weight_sum > 0 else 0
        step += 1
        _log(
            f"  step {step}/{num_steps}: "
            f"loss:mean={loss_mean:.6f}, grad_norm={grad_norm:.4f}, lr={lr:.2e}"
        )

    print(
        f"\nTraining complete. Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB"
    )

    # -- Save adapter + rename lm_head keys (identical on both sides) --
    assert 0.0 < POST_TRAIN_DELTA_SCALE <= 1.0
    raw_delta_sq = 0.0
    scaled_delta_sq = 0.0
    with torch.no_grad():
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            initial = initial_trainable[name].to(
                device=param.device,
                dtype=param.dtype,
            )
            learned_delta = param.data - initial
            raw_delta_sq += float(
                learned_delta.float().pow(2).sum().cpu()
            )
            param.data.copy_(
                initial + POST_TRAIN_DELTA_SCALE * learned_delta
            )
            retained_delta = param.data - initial
            scaled_delta_sq += float(
                retained_delta.float().pow(2).sum().cpu()
            )
    raw_delta_norm = math.sqrt(raw_delta_sq)
    scaled_delta_norm = math.sqrt(scaled_delta_sq)
    assert raw_delta_norm > 0.0, "Training produced a zero adapter delta"
    expected_scaled_norm = POST_TRAIN_DELTA_SCALE * raw_delta_norm
    assert math.isclose(
        scaled_delta_norm,
        expected_scaled_norm,
        rel_tol=2e-3,
        abs_tol=1e-8,
    ), (scaled_delta_norm, expected_scaled_norm)
    print(
        f"Post-train delta scaling: {POST_TRAIN_DELTA_SCALE:.2f}; "
        f"raw_norm={raw_delta_norm:.6f}, "
        f"retained_norm={scaled_delta_norm:.6f}"
    )

    from safetensors.torch import load_file, save_file

    save_dir = "." if IS_KAGGLE else OUTPUT_DIR
    os.makedirs(save_dir, exist_ok=True)
    for _f in os.listdir(save_dir):
        if _f.startswith("adapter"):
            os.remove(os.path.join(save_dir, _f))
    model.save_pretrained(save_dir)
    st_path = os.path.join(save_dir, "adapter_model.safetensors")
    tensors = load_file(st_path)
    renamed = {
        k.replace("base_model.model.lm_head.", "base_model.model.backbone.lm_head."): v
        for k, v in tensors.items()
    }
    save_file(renamed, st_path)

    # -- Clean unsloth compiled cache (runs on both) ------------------
    _ucache = "unsloth_compiled_cache"
    if os.path.isdir(_ucache):
        import shutil as _sh

        _sh.rmtree(_ucache)

    # -- Package & ship (divergent) -----------------------------------
    if IS_KAGGLE:
        import zipfile

        adapter_files = [f for f in os.listdir(save_dir) if f.startswith("adapter")]
        SUBMISSION_ZIP = "submission.zip"
        with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in adapter_files:
                zf.write(os.path.join(save_dir, fname), fname)
        for fname in adapter_files:
            os.remove(os.path.join(save_dir, fname))
        print(f"Wrote {SUBMISSION_ZIP}")
        with zipfile.ZipFile(SUBMISSION_ZIP, "r") as _zf:
            zip_names = set(_zf.namelist())
            assert zip_names == {
                "adapter_config.json",
                "adapter_model.safetensors",
            }, zip_names
            saved_config = json.loads(_zf.read("adapter_config.json"))
            assert saved_config.get("r") == LORA_RANK, saved_config
            assert saved_config.get("lora_alpha") == LORA_ALPHA, saved_config
        print(
            "Submission validation OK: root files only, "
            f"rank={LORA_RANK}, alpha={LORA_ALPHA}"
        )
    else:  # IS_MODAL_WORKER
        import shutil
        import tempfile

        with open(os.path.join(save_dir, "training_log.txt"), "w") as f:
            f.write("\n".join(training_log) + "\n")
        output_vol.commit()  # noqa: F821 - defined at module level on non-Kaggle

        kaggle_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(os.path.join(kaggle_dir, "access_token"), "w") as f:
            f.write(os.environ["KAGGLE_API_TOKEN"])
        upload_dir = tempfile.mkdtemp()
        for fname in os.listdir(save_dir):
            shutil.copy(os.path.join(save_dir, fname), upload_dir)
        metadata = {"id": KAGGLE_DATASET, "title": KAGGLE_DATASET.split("/")[1]}
        with open(os.path.join(upload_dir, "dataset-metadata.json"), "w") as f:
            json.dump(metadata, f)
        print(f"Uploading to Kaggle {KAGGLE_DATASET}...")
        subprocess.run(
            [
                "kaggle",
                "datasets",
                "version",
                "-p",
                upload_dir,
                "-m",
                "post-finetuned adapter + compiled wheels",
            ],
            check=True,
        )
        print("Kaggle upload complete.")
    print("Training complete.")


In [ ]:
# -- Modal glue: image, app, volumes, train_remote, main --------------
# Defined at module level on non-Kaggle so the worker's module import
# registers train_remote with the app. On Kaggle, skipped entirely
# (modal package is not installed there).
if not IS_KAGGLE:
    import modal

    train_image = (
        modal.Image.from_registry(
            "nvidia/cuda:12.8.1-devel-ubuntu22.04",
            add_python="3.12",
        )
        .entrypoint([])
        .apt_install("git", "build-essential", "clang")
        .pip_install(
            "torch==2.10.0",
            extra_index_url="https://download.pytorch.org/whl/cu128",
        )
        .pip_install(
            "safetensors>=0.5.0",
            "transformers>=4.56.2",
            "accelerate>=1.0.0",
            "peft>=0.15.0",
            "bitsandbytes>=0.45.0",
            "huggingface_hub>=0.36.2",
            "hf-transfer>=0.1.9",
            "numpy",
            "pillow",
            "torchvision",
            "datasets",
            "sentencepiece",
            "xformers",
            "cut-cross-entropy>=25.1.0",
            "wheel",
            "setuptools",
            "trl",
            "kaggle>=1.6.0",
        )
        .run_commands(
            'python -c "import torch.utils.cpp_extension as e; p=e.__file__; '
            "t=open(p).read().replace('raise RuntimeError(CUDA_MISMATCH_MESSAGE', 'pass  # '); "
            "open(p,'w').write(t)\"",
            "TORCH_CUDA_ARCH_LIST='12.0' pip wheel --no-build-isolation --wheel-dir /wheels mamba_ssm==2.3.1 causal_conv1d==1.6.1",
            "pip install --no-deps /wheels/mamba_ssm-*.whl /wheels/causal_conv1d-*.whl",
            "pip install --no-deps 'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo'",
            "pip install --no-deps 'unsloth[base] @ git+https://github.com/unslothai/unsloth'",
        )
        .pip_install("einops")
        .env({"HF_HOME": "/root/.cache/huggingface"})
    )

    hf_cache_vol = modal.Volume.from_name("huggingface-cache", create_if_missing=True)
    merged_vol = modal.Volume.from_name("merged-adapter", create_if_missing=True)
    corpus_vol = modal.Volume.from_name("corpus-data", create_if_missing=True)
    output_vol = modal.Volume.from_name("post-finetune-output", create_if_missing=True)

    app = modal.App("post-finetune-pipeline")

    @app.function(
        image=train_image,
        gpu="RTX-PRO-6000",
        volumes={
            "/root/.cache/huggingface": hf_cache_vol,
            "/merged": merged_vol,
            "/data": corpus_vol,
            "/output": output_vol,
        },
        timeout=6 * 60 * MINUTES,
        secrets=[modal.Secret.from_local_environ(["KAGGLE_API_TOKEN"])],
    )
    def train_remote() -> None:
        run_training()

    if IS_MODAL_LAUNCHER:

        @app.local_entrypoint()
        def main() -> None:
            train_remote.remote()

In [ ]:
# On Kaggle, trigger training directly after cells load.
# On Modal worker, Modal's runtime calls train_remote() which calls run_training().
# On Modal launcher, neither fires (main() submits the remote call instead).
if IS_KAGGLE:
    run_training()